# Cross-Screen Measurement: Conversion Drivers (PRACTICE)

This is the **do-it-yourself** version of `measurement_drivers.ipynb`.
The explanations stay; the PySpark is gone. Your job is to write the code in
each `YOUR TASK` cell, then run the `SELF-CHECK` cell under it to see PASS or the
number you were supposed to hit.

**How to practice (the rules that make it work):**
- Do not open the finished notebook until you are genuinely stuck. Recognition is
  not recall; the point is to make the answer come from your hands.
- On the judgment cells (the naive join, the weighted regression), **predict the
  number before you run it.** Being wrong on the prediction is where the learning is.
- The finished `measurement_drivers.ipynb` in this folder is your answer key.

Run the two setup cells below as-is (Spark session and config are boilerplate, not
the thing you are practicing), then start at Step 1.

In [ ]:
# --- GIVEN: config (do not edit) ---
DATA = "data"
CAMPAIGN_ID = "CMP_2026_Q2_AUTO"
MODEL_FORMULA = "converted ~ exposure_frequency + viewing_profile + segment"
WEIGHT_COL = "panel_weight"
import os, subprocess, sys
if DATA == "data" and not os.path.exists(os.path.join(DATA, "panel.csv")):
    subprocess.run([sys.executable, "generate_sample_data.py"], check=True)

In [ ]:
# --- GIVEN: Spark session (do not edit) ---
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = (SparkSession.builder.appName("samba-practice").master("local[*]")
             .config("spark.driver.host", "localhost")
             .config("spark.sql.shuffle.partitions", "4").getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
from pyspark.sql import functions as F, Window
print("Spark", spark.version, "ready")

## 1. Retrieve

Get the three CSVs into one engine. `inferSchema` types the columns so
`exposure_frequency` arrives as an integer.

In [ ]:
# YOUR TASK: read the three CSVs into DataFrames named exposure, conv, panel.
# Files: {DATA}/ad_exposure.csv, {DATA}/conversions.csv, {DATA}/panel.csv
# Hint: spark.read.csv(path, header=True, inferSchema=True)

# exposure = ...
# conv     = ...
# panel    = ...

In [ ]:
# --- SELF-CHECK ---
try:
    assert exposure.count() == 3232, f"exposure should be 3232 rows, got {exposure.count()}"
    assert conv.count() == 572,      f"conversions should be 572 rows, got {conv.count()}"
    assert panel.count() == 4000,    f"panel should be 4000 rows, got {panel.count()}"
    print("PASS: 3232 exposure, 572 conversions, 4000 panel")
except Exception as e:
    print("Not passing yet:", type(e).__name__, "-", e)

## 2. Link (the naive attempt)

The `household_id` key drifts across sources (`HH000123`, `hh000123`, `  HH000123 `).
Join exposure to panel on the **raw** key and measure how many exposure rows the
join silently drops.

**Predict first:** of 3232 exposure rows, how many do you think survive an inner
join on the unclean key? Write your guess down, then run the check.

In [ ]:
# YOUR TASK: inner-join panel and exposure on the raw "household_id".
# Compute: matched = rows in the join, dropped = exposure rows that did NOT match.
# Hint: panel.join(exposure, on="household_id", how="inner")

# naive = ...
# matched = ...
# dropped = ...

In [ ]:
# --- SELF-CHECK ---
try:
    assert matched == 2256, f"inner join should match 2256, got {matched}"
    assert dropped == 976,  f"should drop 976 rows, got {dropped}"
    print(f"PASS: {matched} matched, {dropped} silently lost to key drift")
except Exception as e:
    print("Not passing yet:", type(e).__name__, "-", e)

## 3. Shape

Fix the three defects: (1) normalize every key with `UPPER(TRIM(...))`; (2) drop
the double-logged duplicate rows; (3) fill blank devices with `Unknown` and blank
segments with `Unclassified`. **Watch the null trap:** empty CSV fields read as
true `NULL`, so a `== ""` test misses them. Test `isNull()` explicitly.

In [ ]:
# YOUR TASK:
#  - write clean_key(df) that upper-trims household_id
#  - apply it to all three -> exposure_c, conv_c, panel_c
#  - dedupe exposure_c (dropDuplicates)
#  - on exposure_c: blank/null primary_device -> "Unknown"
#  - on panel_c:    blank/null segment        -> "Unclassified"
# Hint for nulls: F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), "X").otherwise(F.col(c))

# def clean_key(df): ...
# exposure_c, conv_c, panel_c = ...
# exposure_c = exposure_c.dropDuplicates()
# ...null handling...

In [ ]:
# --- SELF-CHECK ---
try:
    n = exposure_c.count()
    assert 3100 <= n <= 3200, f"after dedup expect ~3150 rows, got {n}"
    seg_vals = [r["segment"] for r in panel_c.select("segment").distinct().collect()]
    assert "Unclassified" in seg_vals, "blank segments should become 'Unclassified'"
    assert all(s == s.upper().strip() for s in
               [r["household_id"] for r in exposure_c.select("household_id").limit(50).collect()]), \
           "keys not normalized to UPPER/TRIM"
    print(f"PASS: deduped to {n} rows, keys normalized, segments cleaned")
except Exception as e:
    print("Not passing yet:", type(e).__name__, "-", e)

## 4. Link (correct)

Build the `master` household table. Two design choices that carry meaning: the
**panel is the spine** (left base) so zero-exposure households survive, and
conversions collapse to a **distinct flag** before joining so no household
double-counts.

In [ ]:
# YOUR TASK: build master =
#   panel_c  LEFT JOIN  exposure_c[household_id, exposure_frequency, primary_device]
#            (fill missing exposure_frequency with 0, primary_device with "Unknown")
#   then LEFT JOIN a distinct conversion flag (converted=1), fill non-converters with 0.
# Hint: conv_flag = conv_c.select("household_id").distinct().withColumn("converted", F.lit(1))

# master = ...

In [ ]:
# --- SELF-CHECK ---
try:
    assert master.count() == 4000, f"master should be 4000 households, got {master.count()}"
    c = master.filter("converted = 1").count()
    assert c == 572, f"should be 572 converted, got {c}"
    print("PASS: 4000 households, 572 converted, zero-exposure HHs retained")
except Exception as e:
    print("Not passing yet:", type(e).__name__, "-", e)

## 5. Reduce

Two reductions. First: conversion rate by segment **weighted by panel_weight**,
ranked with a window function. Second: the conversion-vs-frequency curve that
shows saturation.

In [ ]:
# YOUR TASK 1: build seg with columns
#   segment, households (count), avg_freq (avg exposure_frequency rounded 2),
#   weighted_cvr = sum(converted*panel_weight) / sum(panel_weight) rounded 4,
#   and a "rank" via Window.orderBy(weighted_cvr desc) + row_number().
# YOUR TASK 2: build freq grouping by exposure_frequency with cvr = avg(converted).

# seg = ...
# freq = ...
# seg.show(truncate=False); freq.orderBy("exposure_frequency").show(8)

In [ ]:
# --- SELF-CHECK ---
try:
    top = seg.orderBy(F.col("weighted_cvr").desc()).first()
    assert top["segment"] == "Cordcutter_HHI_High", f"top segment should be Cordcutter_HHI_High, got {top['segment']}"
    assert 0.18 <= top["weighted_cvr"] <= 0.20, f"top weighted_cvr ~0.1915, got {top['weighted_cvr']}"
    assert seg.count() == 6, f"expect 6 segments (incl Unclassified), got {seg.count()}"
    print(f"PASS: top segment {top['segment']} at weighted CVR {top['weighted_cvr']}")
except Exception as e:
    print("Not passing yet:", type(e).__name__, "-", e)

## 6. Infer

Fit a **weighted logistic regression** so the coefficients estimate each factor's
effect holding the others constant. Use `weightCol=panel_weight` so estimates are
population-representative. The feature-name-to-coefficient mapping below is fiddly
metadata plumbing, so it is **given**; you write the model.

**Predict first:** which factor will have the largest positive coefficient?

In [ ]:
# YOUR TASK:
#  - model_df = master with segment != "Unclassified"
#  - RFormula(formula=MODEL_FORMULA, featuresCol="features", labelCol="label"),
#    fit on model_df, transform -> trans
#  - LogisticRegression(featuresCol="features", labelCol="label",
#       weightCol=WEIGHT_COL, maxIter=50), fit on trans -> lr_model
from pyspark.ml.feature import RFormula
from pyspark.ml.classification import LogisticRegression

# model_df = ...
# rf = ...; rf_model = ...; trans = ...
# lr = ...; lr_model = ...

In [ ]:
# --- GIVEN: map coefficients to names, then rank (run after your model fits) ---
try:
    meta = trans.schema["features"].metadata["ml_attr"]
    names = [None] * meta["num_attrs"]
    for group in meta["attrs"].values():
        for a in group:
            names[a["idx"]] = a["name"]
    drivers = sorted(zip(names, lr_model.coefficients), key=lambda x: abs(x[1]), reverse=True)
    auc = lr_model.summary.areaUnderROC
    print(f"AUC: {auc:.3f}")
    for n, c in drivers:
        print(f"  {c:+.4f}  {n}")
    # SELF-CHECK
    assert 0.60 <= auc <= 0.75, f"AUC should be ~0.68, got {auc:.3f}"
    assert drivers[0][0] == "segment_Cordcutter_HHI_High", f"top driver should be Cordcutter_HHI_High, got {drivers[0][0]}"
    print("\nPASS: model recovers the planted signal (AUC ~0.68, top driver Cordcutter_HHI_High)")
except Exception as e:
    print("Not passing yet:", type(e).__name__, "-", e)

## 7. Translate

Move the small reduced frames to pandas and draw the two pictures: the segment
ranking and the saturation curve.

In [ ]:
# YOUR TASK: seg_pd = seg.toPandas(); freq_pd = freq.toPandas()
# Make a 2-panel matplotlib figure (barh of weighted_cvr by segment; line of cvr vs
# exposure_frequency) and save to f"{DATA}/practice_chart.png".
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

# seg_pd, freq_pd = ...
# fig, ax = plt.subplots(1, 2, figsize=(12, 4.5)) ...
# plt.savefig(f"{DATA}/practice_chart.png", dpi=110)

In [ ]:
# --- SELF-CHECK ---
try:
    assert len(seg_pd) == 6, f"seg_pd should have 6 rows, got {len(seg_pd)}"
    assert os.path.exists(f"{DATA}/practice_chart.png"), "chart file not written"
    print("PASS: frames in pandas, chart written. You rebuilt the whole pipeline.")
except Exception as e:
    print("Not passing yet:", type(e).__name__, "-", e)

## You're done when every cell says PASS

If all seven self-checks pass, you wrote the entire measurement pipeline from
blanks: retrieve, link, shape, link-correct, reduce, infer, translate. That is the
artifact you point at in an interview, and now the code came from you.

**Next level (no answer key for these):**
- Add `primary_device` to `MODEL_FORMULA` and see whether it earns a real coefficient.
- Rebuild the weighted_cvr **without** panel weights and explain what changed and why.
- Replace the `row_number()` rank with `dense_rank()` and articulate the difference.